# [LAB] LLM API 실습

같은 질문 "LLM이 무엇인지 3문장으로 설명해줘."를 세 가지 방법으로 LLM에 보내 봅니다. 방법이 바뀔 때마다 무엇이 편해지는지 비교하는 것이 이 노트북의 목표입니다.

- **2장 요청을 직접 보내기**: 주소·헤더·요청 본문을 손으로 만들어 보냅니다. 코드는 이미 완성되어 있으니 실행하고, CodeMate에게 설명을 받습니다.
- **3장 SDK로 묻기**: 도구(SDK)가 요청을 대신 보냅니다. 한 번 묻기, 이어서 묻기, 조각으로 받기를 실행하고 CodeMate에게 설명을 받습니다.
- **4장 LangChain으로 호출하기**: 코드 셀의 빈칸(✍ 표시 줄)을 CodeMate와 함께 채우고 실행한 뒤, 접힌 "정답 보기"와 비교합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**목차**

- 1. 실습 방법 (실습)
- 2. 요청을 직접 보내기 (코드) (실습)
  - 2.1 요청 본문 만들기 (코드)
  - 2.2 보내고 응답 받기 (코드)
  - 2.3 참고: 다른 요청 방식(Responses API)
  - 2.4 CodeMate로 설명 받기 (실습)
- 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기 (코드) (실습)
  - 3.1 한 번 묻기 (코드)
  - 3.2 이어서 묻기 (코드)
  - 3.3 조각으로 받기 (코드)
  - 3.4 참고: temperature와 top_p
  - 3.5 CodeMate로 설명 받기 (실습)
- 4. LangChain으로 호출하기 (코드) (실습)
  - 4.1 역할 메시지로 한 번에 받기 (실습)
  - 4.2 조각으로 받기 (실습)
- 5. 정리

In [ ]:
import sys

sys.path.insert(0, '..')  # lessons/assets 도구를 불러오기 위한 경로

from assets.widgets import show_checklist

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 읽습니다

# 값은 출력하지 않습니다. 빠진 설정이 있는지만 봅니다.
provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
필요한_이름 = {
    "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
    "samsung": ["api_base_url", "model", "credential_key", "send_system_name", "user_id"],
}.get(provider, ["LLM_PROVIDER"])
빠진_이름 = [name for name in 필요한_이름 if not os.getenv(name)]

import config.llm  # 실습 코드가 쓰는 설정 모듈을 불러옵니다

if 빠진_이름:
    print(f"준비가 덜 되었습니다. 빠진 설정 {len(빠진_이름)}개가 있습니다. 강사에게 알려 주세요.")
    print("(강사 확인용:", ", ".join(빠진_이름), ")")
else:
    print("준비 완료: LLM에 연결할 설정이 모두 있습니다.")

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/01-llm-api/01_requests.py`·`02_openai.py`·`03_langchain.py`. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다.
- 원본에는 환경에 따라 주소·모델·인증을 고르는 분기가 있습니다(`LLM_PROVIDER`). openai 분기는 사외 준비용으로 우리가 더한 것이라 수강생 글에서는 다루지 않습니다. 2장의 보이는 셀(요청 본문)에 `provider`·`gpt-6-luna` 분기가 그대로 보이니, 물으면 "연결 환경에 맞춘 설정"이라고만 답합니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지, 셀을 직접 고칠 수 있는지는 강의장에서 아직 확인하지 않았습니다. 못 읽으면 셀 코드를 지시문 아래에 붙여 넣게 하고, 못 고치면 채팅 답의 코드를 수강생이 빈칸에 붙여 넣게 합니다.

</details>

## 1. 실습 방법 <small>(실습)</small>

CodeMate에 주는 지시문은 다섯 칸으로 씁니다. 무엇을 원하는지, 어떤 조건인지, 어떤 모양으로 받을지, 헷갈릴 점과 하지 말 것을 미리 적어 두면 원하는 코드를 받기 쉽습니다.

```text
[역할·목표] 무엇을 하는 사람으로서, 무엇을 만들어 달라는지
[배경·제약] 어느 셀의 어느 빈칸인지, 쓸 수 있는 것과 없는 것
[출력 형식] 코드만? 코드 + 설명 3줄?
[모호함 줄이기] 헷갈릴 만한 부분을 미리 정해 주기
[하지 말 것] 빈칸 밖은 고치지 않기, 새 기능 추가하지 않기
```

순서는 늘 같습니다.
1. 빈칸 위의 TODO 주석을 읽고, 무엇을 채워야 하는지 한 문장으로 말해 봅니다.
2. 빈칸이 든 코드 셀을 선택한 상태에서 지시문을 CodeMate 채팅에 붙여 넣습니다.
3. CodeMate가 채운 코드를 읽고, 모르는 줄이 있으면 "이 줄이 하는 일"을 다시 묻습니다.
4. 실행해 봅니다. 오류가 나면 오류 메시지를 그대로 CodeMate에 붙여 넣고 원인부터 설명해 달라고 합니다.
5. 셀 아래 "정답 보기"를 펼쳐 비교합니다. 다르다고 틀린 것은 아닙니다. 왜 다른지 말할 수 있으면 됩니다.

## 2. 요청을 직접 보내기 <small>(코드) (실습)</small>

`requests`라는 도구로 LLM 서버 주소에 요청을 직접 보냅니다.
요청 본문(payload)은 JSON(이름과 값을 짝지어 적는 데이터 형식)으로 만들고, 응답도 JSON으로 받습니다.

코드는 완성되어 있습니다. 준비 코드부터 차례로 실행하고, 출력된 응답 JSON을 읽어 봅니다. 요청 본문 셀 아래쪽의 `if` 부분은 연결 환경에 맞춰 설정을 조금 바꾸는 코드라 건너뛰어도 됩니다.

**왜 배우나:** 3장의 SDK와 4장의 LangChain도 속에서는 이런 요청을 보냅니다. 날것의 요청을 한 번 보아 두면, API 오류가 났을 때 어디부터 살펴야 할지 알 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고, 요청을 보낼 주소·모델 이름·인증 정보를 준비합니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

provider = os.getenv('LLM_PROVIDER', 'samsung').strip().lower()
if provider == 'openai':
    api_base_url = 'https://api.openai.com/v1/chat/completions'
    model = os.getenv('OPENAI_MODEL')
    api_key = os.getenv('OPENAI_API_KEY')
    if not model or not api_key:
        raise RuntimeError('OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.')
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
elif provider == 'samsung':
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config
    samsung = llm_config()
    api_base_url = f'{samsung["base_url"].rstrip("/")}/chat/completions'
    model = samsung['model']
    headers = {**samsung['headers'], 'Content-Type': 'application/json'}
else:
    raise ValueError(f'지원하지 않는 LLM_PROVIDER: {provider}')

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/01_requests.py`. 보이는 셀은 원본 30~54줄, 참고 코드는 57~83줄입니다.

</details>

### 2.1 요청 본문 만들기 <small>(코드)</small>

In [ ]:
payload = {
    "model": model,
    "messages": [
        { "role": "user","content": "LLM이 무엇인지 3문장으로 설명해줘." },
    ],
    "temperature": 0.5,
    "stream": False,
}
if provider == 'samsung':
    payload['include_reasoning'] = False
elif model == 'gpt-6-luna':
    # GPT-6 Luna의 Chat Completions는 기본 추론 설정에서 temperature를 받지 않는다.
    payload.pop('temperature')
    payload['reasoning_effort'] = 'none'

### 2.2 보내고 응답 받기 <small>(코드)</small>

In [ ]:
new_response = requests.post(
    api_base_url, headers=headers, json=payload, timeout=30
)
new_response.raise_for_status()
recv_json = new_response.json()
  
print("### 응답 Json")
pprint(recv_json)
print("### 결과 메시지")
print(recv_json["choices"][0]["message"]["content"])

### 2.3 참고: 다른 요청 방식(Responses API)

문자열로 감싸 두어 실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# OpenAI Responses API 엔드포인트
# - 기존 Chat Completions(/v1/chat/completions)의 후속 API
# - input 필드 하나로 단순 문자열과 대화 내역 모두 처리 가능
api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

if not api_key:
    raise RuntimeError('OPENAI_API_KEY 환경변수를 설정해주세요.')

url = 'https://api.openai.com/v1/responses'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

payload = {
    'model': model,
    'input': 'LLM이 무엇인지 3문장으로 설명해줘.',
}

response = requests.post(url, headers=headers, json=payload, timeout=30)
response.raise_for_status()

data = response.json()
pprint(data)
```

**볼 지점**
- "### 결과 메시지" 아래 줄이 모델의 답변 텍스트입니다.
- "### 응답 Json" 아래에서 `choices` → `message` → `content`를 따라가면 같은 텍스트가 있습니다. 코드는 이 경로로 답을 꺼냈습니다.
- `usage`를 찾아봅니다. `prompt_tokens`는 입력 토큰 수, `completion_tokens`는 출력 토큰 수, `total_tokens`는 둘의 합입니다.
- 숫자는 실행할 때마다 다릅니다. `usage`가 보이지 않으면 "이 환경에서는 제공되지 않음"이라고 적어 둡니다.

<details><summary>강사용 메모</summary>

- 원본 9~28줄의 분기에서 `.env`의 `LLM_PROVIDER` 값에 따라 주소·모델·헤더가 정해집니다. 기본값은 samsung입니다.
- 원본 38~43줄(요청 본문 셀 아래쪽 if): samsung이면 `include_reasoning=False`를 더하고, openai이면서 모델이 `gpt-6-luna`이면 temperature를 빼고 `reasoning_effort='none'`을 넣습니다.
- 사내 게이트웨이 응답에 `usage`가 들어오는지는 아직 확인하지 않았습니다. 토큰 수는 환경마다 다릅니다.

</details>

### 2.4 CodeMate로 설명 받기 <small>(실습)</small>

**CodeMate 지시문**

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람을 돕는 코드 해설자입니다.
목표는 제가 아래에 붙여 넣은 노트북 코드 셀 두 개(요청 본문 만들기, 보내고 응답 받기)를 이해하는 것입니다.

[배경·제약]
- 이 코드는 LLM 서버에 요청을 보내고, 응답 JSON과 답변 텍스트를 출력합니다.
- 저는 두 셀을 이미 실행했고, 출력 결과를 가지고 있습니다.
- 붙여 넣은 두 셀만 다룹니다. 요청 본문 아래의 if 부분은 연결 환경에 맞춰 설정을 바꾸는 코드이니 한 줄로만 요약해 주세요.

[출력 형식]
1. 두 셀의 코드를 "코드 줄 | 쉬운 한국어 설명 1문장" 표로 써 주세요. 빈 줄은 빼 주세요.
2. pprint로 출력하는 응답 JSON의 usage 부분에 어떤 이름의 값이 나올지 예측해 목록으로 써 주세요.
   숫자는 맞히지 말고, 각 값이 무엇을 세는지 한 문장씩 붙여 주세요.

[모호함 줄이기]
- 전문 용어는 처음 나올 때 괄호 안에 쉬운 말로 풀어 주세요.
- 확실하지 않은 것은 "확실하지 않음"이라고 써 주세요.

[하지 말 것]
- 코드를 새로 쓰거나 고치지 마세요.
- 더 좋은 코드나 리팩터링을 제안하지 마세요.

마지막에 왜 이렇게 설명했는지 3줄로 써 주세요.

(여기 아래에 두 셀의 코드를 복사해 붙여 넣습니다)
```

**볼 지점**
- CodeMate가 예측한 `usage`의 이름들을 위 실행 결과의 `usage`와 하나씩 맞춰 봅니다.
- 표의 설명 중 실행 결과와 맞지 않는 줄이 있으면, 그 코드 줄을 그대로 들어 다시 묻습니다.
- CodeMate가 코드를 고치자고 하면 지시문의 [하지 말 것]을 다시 붙여 넣습니다.

In [ ]:
show_checklist(['오류 없이 실행되어 답변 텍스트가 출력되었다',
 '응답 JSON에서 답변 텍스트가 들어 있는 경로를 찾았다',
 'usage의 세 값이 각각 무엇을 세는지 말할 수 있다',
 'CodeMate의 설명을 실행 결과와 대조했다'], title='요청을 직접 보내기 리뷰')

## 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기 <small>(코드) (실습)</small>

이번에는 OpenAI SDK(개발 도구 묶음)로 같은 일을 합니다. 2장에서 손으로 만든 주소·헤더·JSON 처리를 `client`(요청을 대신 보내 주는 객체)가 맡습니다. 요청 방식은 `messages` 대신 `input`에 내용을 담는 Responses API(새 요청 방식)입니다.

한 번 묻기(Single Turn), 이어서 묻기(Multi Turn), 조각으로 받기(Streaming)를 차례로 실행합니다. 요청을 네 번 보내므로 답이 모두 나오기까지 조금 걸립니다.

**왜 배우나:** 모델은 앞 대화를 기억하지 않습니다. "이어서 묻기" 셀에서 코드가 이전 대화를 매번 다시 넣어 보내는 모습을 직접 봅니다. 메모리나 Checkpointer 같은 기능은 이 "다시 넣기"를 도구가 대신 해 주는 방법입니다.

이 장의 준비 코드입니다. 요청을 대신 보내 줄 `client`와 모델 이름을 만듭니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
import os
from collections import Counter
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

provider = os.getenv('LLM_PROVIDER', 'samsung').strip().lower()
if provider == 'openai':
    model = os.getenv('OPENAI_MODEL')
    api_key = os.getenv('OPENAI_API_KEY')
    if not model or not api_key:
        raise RuntimeError('OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.')
    client = OpenAI(api_key=api_key)
elif provider == 'samsung':
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config
    samsung = llm_config()
    model = samsung['model']
    client = OpenAI(
        api_key='samsung-gateway',  # 실제 사내 인증은 x-dep-ticket 헤더에서 처리
        base_url=samsung['base_url'],
        default_headers=samsung['headers'],
    )
else:
    raise ValueError(f'지원하지 않는 LLM_PROVIDER: {provider}')


# OpenAI SDK 예제는 사외에서는 공식 API, 사내에서는 호환 게이트웨이를 사용한다.

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/02_openai.py`. 한 번 묻기 34~42줄, 이어서 묻기 44~58줄, 조각으로 받기 61~74줄, 참고 77~89줄.

</details>

### 3.1 한 번 묻기 <small>(코드)</small>

In [ ]:
# ── Single Turn ───────────────────────────────────────────────────────────────
# input에 문자열을 직접 전달하면 사용자 메시지로 처리
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
)
print(response.output_text)
print()

### 3.2 이어서 묻기 <small>(코드)</small>

In [ ]:
# ── Multi Turn ────────────────────────────────────────────────────────────────
# input에 역할(role)이 있는 메시지 리스트를 전달하면 대화 맥락 유지
# developer: 시스템 지시 / user: 사용자 입력 / assistant: AI 응답
context = [
    {'role': 'developer', 'content': '너는 단답형 답변 챗봇이야. 경어체를 유지하면서, 가능하면 단답형으로 답변해줘.'},
    {'role': 'user', 'content': '대한민국의 수도는 어디지?'},
]
response_1 = client.responses.create(model=model, input=context)
print(response_1.output_text)

# 이전 응답(response_1.output)을 context에 추가해 대화를 이어감
context += response_1.output
context += [{'role': 'user', 'content': '그곳의 인구는 어느정도야?'}]
response_2 = client.responses.create(model=model, input=context)
print(response_2.output_text)

### 3.3 조각으로 받기 <small>(코드)</small>

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()을 사용하면 LLM이 생성하는 토큰을 실시간으로 받을 수 있음
# output_text.delta 이벤트에 실제 텍스트 조각이 담겨 있음
with client.responses.stream(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
) as stream:
    event_types = []
    for event in stream:
        event_types.append(event.type)
        if 'output_text.delta' in event.type:
            print(event.delta, end='')
    print()
    pprint(Counter(event_types))

### 3.4 참고: temperature와 top_p

문자열로 감싸 두어 실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# temperature: 0에 가까울수록 일관된 답변, 높을수록 창의적(최대 2.0)
# top_p: 확률 상위 p% 토큰만 사용. temperature와 동시 조정은 권장하지 않음
# 주의: gpt-5 계열은 temperature 미지원
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
    temperature=0.2,
    top_p=0.1,
)
print(response.output_text)
```

**볼 지점**
- 첫 답은 2장과 같은 질문에 대한 답입니다. 2장의 답과 문장이 똑같은지 비교해 봅니다.
- "대한민국의 수도는 어디지?"에 짧은 존댓말로 답했는지 봅니다. 이 규칙은 `developer` 메시지에 들어 있습니다.
- "그곳의 인구는 어느정도야?"에 서울의 인구로 답했는지 봅니다. 앞 대화를 `context`에 붙여 다시 보냈기 때문입니다.
- 마지막 답은 조각으로 이어서 출력됩니다. 맨 아래 `Counter(...)`에서 `response.output_text.delta` 옆의 수가 텍스트 조각 이벤트가 온 횟수입니다.

<details><summary>강사용 메모</summary>

- 사내 게이트웨이가 Responses API(`client.responses`)를 지원하는지는 아직 확인하지 않았습니다. 사내에서 오류가 나면 CodeMate 설명만 하고 넘어갑니다.
- 사내 경로에서는 인증을 헤더로 하므로 `api_key` 자리에 이름만 채웁니다(원본 23줄, 접힌 준비 코드 안).

</details>

### 3.5 CodeMate로 설명 받기 <small>(실습)</small>

**CodeMate 지시문**

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람을 돕는 코드 해설자입니다.
목표는 제가 아래에 붙여 넣은 "이어서 묻기" 셀의 코드를 이해하는 것입니다.

[배경·제약]
- 이 셀은 질문을 두 번 보냅니다. 두 번째 질문 "그곳의 인구는 어느정도야?"의 "그곳"은 앞 대화에 나온 곳입니다.
- 저는 이 셀을 이미 실행했고, 두 번째 답이 무엇이었는지 알고 있습니다.
- 붙여 넣은 셀만 다룹니다.

[출력 형식]
1. 첫 번째 요청과 두 번째 요청에 각각 무엇이 담겨 보내지는지 순서대로 목록으로 써 주세요.
2. 모델이 "그곳"을 알아듣는 이유를 2문장으로 써 주세요.
3. context += response_1.output 줄을 지우면 두 번째 답이 어떻게 달라질지 예측해 1문장으로 써 주세요.

[모호함 줄이기]
- "기억한다"는 표현을 쓰려면 모델이 기억하는 것인지, 코드가 다시 보내는 것인지 구분해 주세요.
- 확실하지 않은 것은 "확실하지 않음"이라고 써 주세요.

[하지 말 것]
- 코드를 고치거나 실행하지 마세요.
- 다른 셀(한 번 묻기, 조각으로 받기)은 설명하지 마세요.

마지막에 왜 이렇게 설명했는지 3줄로 써 주세요.

(여기 아래에 "이어서 묻기" 셀의 코드를 복사해 붙여 넣습니다)
```

**볼 지점**
- CodeMate가 쓴 두 번째 요청의 내용 목록에 첫 질문, 첫 답, 새 질문이 모두 들어 있는지 봅니다.
- "모델이 기억해서"라고만 설명했다면, 어느 줄이 대화를 다시 넣는지 물어봅니다.
- 3번 예측은 실행해 보지 않은 추측입니다. 이 노트북에는 그 줄을 지운 경우가 없으므로, 사실로 받아들이지 않습니다.

In [ ]:
show_checklist(['네 번의 답이 모두 오류 없이 출력되었다',
 '두 번째 질문에서 "그곳"을 알아들은 이유를 코드 줄로 가리킬 수 있다',
 '조각으로 받은 답이 이어서 출력되는 것을 확인했다',
 'CodeMate의 설명을 실행 결과와 대조했다'], title='SDK로 한 번 묻기, 이어 묻기, 조각으로 받기 리뷰')

## 4. LangChain으로 호출하기 <small>(코드) (실습)</small>

같은 질문을 LangChain(LLM 호출을 묶어 쓰게 해 주는 도구)으로 보냅니다. 모델 준비는 접힌 준비 코드 안의 `llm = get_llm()` 한 줄로 끝납니다. 2·3장에서 길게 쓰던 준비 코드가 이 함수 하나에 모여 있습니다.

이 장의 코드 셀에는 빈칸(✍ 표시 줄)이 세 곳 있습니다. 메시지 목록 만들기, 한 번에 답 받기(`invoke`), 조각으로 답 받기(`stream`)입니다. 빈칸마다 위의 지시문을 CodeMate에 주고, 돌아온 코드를 읽은 뒤 채워 넣습니다. 셀 하나에 빈칸이 두 곳 있으면 둘 다 채운 다음 실행합니다.

**왜 배우나:** 앞으로 LangChain, RAG, LangGraph로 만드는 에이전트도 모두 이 `get_llm()`과 메시지 목록으로 모델을 호출합니다.

준비 코드는 메시지 종류(`SystemMessage`, `HumanMessage`)를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 펼쳐서 `get_llm()` 줄만 확인해 봐도 좋습니다. 실행을 잊지 않습니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()

# 핵심은 llm 을 사용해서 답변을 만들어 내는 것.
    # ai에게 어떤 지시를 내리고, 답변을 받을 수 있어야 한다는 것.
    # AI에게 어떤 지시를 내린다 -> 채팅창에 시킬일을 적는다.
        # AI API등을 사용할떄는 그 지시조차도 구분해야한다. 사용자가 질문하고 싶은 것, 시스템적으로 지시하고 싶은것 (프롬프트)
llm = get_llm()

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/03_langchain.py`, 정답은 `src/solution/…/03_langchain.py`. `get_llm()`은 `src/config/llm.py`에 있고, `.env`의 `LLM_PROVIDER`에 따라 모델 설정을 골라 돌려줍니다.

</details>

### 4.1 역할 메시지로 한 번에 받기 <small>(실습)</small>

**TODO: SystemMessage와 HumanMessage를 작성해 주세요.**

`messages` 목록에 모델에게 보낼 메시지 두 개를 넣습니다.
하나는 모델의 역할을 정하는 시스템 메시지, 다른 하나는 사용자의 질문입니다.
3장 "이어서 묻기"의 `developer`·`user` 목록과 같은 일을 LangChain의 메시지 종류로 합니다.

**CodeMate 지시문**

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람과 함께 코드를 쓰는 페어 프로그래머입니다.
목표는 제가 선택한 노트북 코드 셀의 첫 번째 빈칸(✍ 표시 줄, messages 목록 안)을 채우는 것입니다.

[배경·제약]
- messages는 LLM에 보낼 대화 목록입니다. 빈칸 자리에 메시지 두 개를 차례로 넣습니다.
  1) 시스템 메시지(모델의 역할): 너는 초보자를 위한 AI 튜터야
  2) 사용자 메시지(질문): LLM이 무엇인지 3문장으로 설명 해 줘.
- 메시지 종류(SystemMessage, HumanMessage)는 앞 셀에서 이미 불러왔습니다. 그것만 씁니다.

[출력 형식]
빈칸 줄을 대신할 코드만 보여 주세요. 제가 읽고 빈칸에 넣겠습니다.

[모호함 줄이기]
- 시스템 메시지가 먼저, 사용자 메시지가 다음입니다.
- 문구는 위에 적은 그대로 씁니다.

[하지 말 것]
- 빈칸 밖의 줄은 고치지 마세요. 이 셀의 두 번째 빈칸도 아직 채우지 마세요.
- import 줄을 더하지 마세요.

마지막에 채운 코드가 하는 일을 3줄로 설명해 주세요.
```

**TODO: invoke()로 메시지를 전달해 응답을 생성해 주세요.**

준비한 `llm`에 `messages`를 한 번에 보내고, 답이 다 만들어지면 받습니다. 받은 응답에서 답변 텍스트만 꺼내 출력합니다.
응답은 텍스트만이 아니라 여러 정보가 든 묶음이라, 텍스트가 든 칸을 골라 꺼내야 합니다.

**CodeMate 지시문**

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람과 함께 코드를 쓰는 페어 프로그래머입니다.
목표는 제가 선택한 노트북 코드 셀의 두 번째 빈칸("invoke()로" TODO 주석 바로 아래 ✍ 표시 줄)을 채우는 것입니다.

[배경·제약]
- 같은 셀 위쪽의 messages와, 앞 셀에서 준비한 llm을 씁니다.
- 할 일은 두 가지입니다.
  1) llm의 invoke()로 messages를 한 번에 보내고, 받은 응답을 response라는 이름에 담습니다.
  2) response에서 답변 텍스트만 출력합니다.

[출력 형식]
빈칸 줄을 대신할 코드만 보여 주세요. 제가 읽고 빈칸에 넣겠습니다.

[모호함 줄이기]
- 응답 객체 전체가 아니라 답변 텍스트만 출력합니다.
- 응답을 한 번에 받는 방식입니다. 조각으로 받는 방식은 쓰지 않습니다.

[하지 말 것]
- 빈칸 밖의 줄은 고치지 마세요. 다음 셀의 빈칸은 아직 채우지 마세요.
- 예외 처리, 재시도 같은 기능을 더하지 마세요.

마지막에 채운 코드가 하는 일을 3줄로 설명해 주세요.
```

In [ ]:
# ── 기본 invoke ───────────────────────────────────────────────────────────────
# SystemMessage: LLM의 역할/행동 지침 설정
# HumanMessage: 사용자 입력
messages = [
    # TODO: SystemMessage와 HumanMessage를 작성해 주세요.
    ...  # ✍ 여기를 CodeMate와 함께 채웁니다
]

# TODO: invoke()로 메시지를 전달해 응답을 생성해 주세요.
...  # ✍ 여기를 CodeMate와 함께 채웁니다

**볼 지점 (첫 번째 빈칸)**
- 목록 안에 메시지가 두 개, 시스템 메시지 → 사용자 메시지 순서로 들어갔는지 봅니다.
- 두 메시지 사이에 쉼표가 있는지 봅니다. 하나만 빠져도 파이썬 오류가 납니다.
- 빈칸 밖의 줄이나 `import`가 바뀌지 않았는지 봅니다. 바뀌었으면 되돌려 달라고 다시 지시합니다.

**볼 지점 (두 번째 빈칸)**
- 응답을 담는 이름이 `response`인지 봅니다. 정답과 비교할 때 이 이름을 그대로 씁니다.
- 출력하는 것이 응답 전체가 아니라 답변 텍스트인지 봅니다. 3줄 설명에 "어느 칸에서 텍스트를 꺼냈는지"가 있는지 확인합니다.
- 헷갈리면 "응답 전체를 출력하면 화면이 어떻게 달라지나요?"라고 다시 묻습니다.

<details><summary>정답 보기</summary>

```python
SystemMessage(content='너는 초보자를 위한 AI 튜터야'),   # 기존에 사용하던 페르소나
HumanMessage(content='LLM이 무엇인지 3문장으로 설명 해 줘.')
```

```python
response = llm.invoke(messages) # AIMessage라는 응답을 뱉어냈으니
print(response.content)
```

</details>

### 4.2 조각으로 받기 <small>(실습)</small>

**TODO: stream()으로 응답을 실시간 출력해 주세요.**

같은 `messages`를 이번에는 조각으로 받습니다. 조각이 올 때마다 그 텍스트를 줄을 바꾸지 않고 바로 이어서 출력합니다.
화면에는 답변이 타자 치듯 한 문단으로 이어져 나와야 합니다. 빈칸 바로 위 주석(`end=''`로 줄바꿈 없이 실시간 출력)이 힌트입니다.

**CodeMate 지시문**

```text
[역할·목표]
당신은 파이썬을 처음 배우는 사람과 함께 코드를 쓰는 페어 프로그래머입니다.
목표는 제가 선택한 노트북 코드 셀의 빈칸(셀 맨 끝 ✍ 표시 줄)을 채우는 것입니다.

[배경·제약]
- 앞 셀들에서 만든 llm과 messages를 그대로 씁니다.
- llm의 stream()으로 messages를 보내고, 돌아오는 조각(chunk)을 하나씩 받습니다.
- 조각마다 그 안의 텍스트만 출력합니다. 줄을 바꾸지 않고 이어 붙이며, 받자마자 화면에 보이게 합니다.
- 조각을 다 받은 뒤 줄바꿈을 한 번 출력합니다.

[출력 형식]
빈칸 줄을 대신할 코드만 보여 주세요. 제가 읽고 빈칸에 넣겠습니다.

[모호함 줄이기]
- 조각 객체 전체가 아니라 조각의 텍스트만 출력합니다.
- 조각을 모아 두었다가 한꺼번에 출력하지 않습니다.

[하지 말 것]
- 빈칸 밖의 줄은 고치지 마세요.
- 새 변수, 함수, 예외 처리를 더하지 마세요.

마지막에 채운 코드가 하는 일을 3줄로 설명해 주세요.
```

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()은 토큰 단위로 응답을 yield함
# end='' 로 줄바꿈 없이 실시간 출력
# TODO: stream()으로 응답을 실시간 출력해 주세요.
...  # ✍ 여기를 CodeMate와 함께 채웁니다

**볼 지점**
- 반복문 안에서 조각의 텍스트만 출력하는지 봅니다. 조각 전체를 출력하면 `content=...`로 시작하는 줄이 조각마다 하나씩 나옵니다.
- 줄을 바꾸지 않는 설정과, 받자마자 보이게 하는 설정이 들어 있는지 봅니다. 3줄 설명에서 각각 어느 부분인지 짚어 달라고 해도 됩니다.
- 반복문이 끝난 뒤 줄바꿈이 한 번 있는지 봅니다.
- 아래 "정답 보기"의 코드는 조각을 통째로 출력합니다. 지시문은 텍스트만 이어서 출력하라고 했으니 둘은 다릅니다. 두 방식을 모두 실행해 화면이 어떻게 다른지 비교해 봅니다.

<details><summary>정답 보기</summary>

```python
for chunk in llm.stream(messages):
    print(chunk.content, end='', flush=True)
print()
```

</details>

**볼 지점**
- 답변이 두 번 출력됩니다. 첫 번째는 `invoke()`로 한 번에 받은 답, 두 번째는 `stream()`으로 조각을 이어 붙인 답입니다.
- 두 번째 답이 한 문단으로 이어져 나오면 지시문대로 채운 것입니다. `content=`가 줄마다 보이면 조각 전체를 출력한 것입니다.
- 답이 초보자에게 설명하는 말투인지 봅니다. 시스템 메시지로 정한 역할이 반영된 것입니다.

In [ ]:
show_checklist(['빈칸만 채웠고 다른 줄은 바꾸지 않았다',
 '답변이 두 번(한 번에, 조각으로) 오류 없이 출력되었다',
 '채운 코드의 각 줄이 하는 일을 말할 수 있다',
 '정답과 다른 코드 줄이 있다면 왜 다른지 설명할 수 있다'], title='LangChain으로 호출하기 리뷰')

<details><summary>강사용 메모</summary>

정답 보기는 솔루션(src/solution)의 조각 텍스트 출력 방식입니다. 받은 원본(src/raw)은 `print(chunk)`로 조각 객체를 통째로 출력합니다.

---

- 받은 원본(src/raw)의 스트리밍 정답은 `print(chunk)`로 조각 전체를 출력하고, "정답 보기"에도 그대로 나옵니다. 지시문은 주석·솔루션(src/solution)처럼 텍스트만 이어서 출력하는 것을 목표로 합니다. 이 차이를 리뷰 재료로 씁니다.
- 첫 번째 빈칸 지시문의 문구는 원본 정답 문자열(마침표 없는 "튜터야", 띄어 쓴 "설명 해 줘.")에 맞췄습니다.
- CodeMate가 셀을 직접 고칠 수 있는지는 미확인이라, 지시문은 "코드만 보여 달라"로 썼습니다.
- 사내 환경에서 조각에 `usage_metadata`가 오는지는 아직 확인하지 않았습니다.

</details>

## 5. 정리

같은 질문을 세 가지 방법으로 보냈습니다.

| 장 | 보내는 방법 | 답변 텍스트를 꺼내는 곳 |
|---|---|---|
| 2. 요청을 직접 보내기 | 주소·헤더·JSON을 직접 만들어 보냄 | `choices[0].message.content` |
| 3. SDK로 묻기 | SDK의 `client`가 대신 보냄 | `output_text` |
| 4. LangChain으로 호출하기 | `get_llm()`으로 준비한 모델에 메시지 목록을 보냄 | `content` |

방법은 달라도 "메시지와 설정값을 보내고, 응답에서 답변 텍스트를 꺼낸다"는 흐름은 같습니다. 앞 대화를 이어 가려면 코드가 이전 대화를 다시 넣어 보내야 한다는 점도 확인했습니다.

CodeMate에게는 역할·배경·출력 형식·모호함·하지 말 것을 나눠 지시했고, 돌아온 코드는 직접 실행하고 정답과 비교해 검토했습니다.